# Q1: Median Inter-Order Gap Analysis

Calculate the **median number of days between consecutive orders** for customers who placed more than one order.

In [1]:
import pandas as pd

# Load only the two columns we need to keep memory usage low
df = pd.read_csv(
    "../data/raw/orders.csv",
    usecols=["customer_id", "order_date"]
)

# Convert order_date to proper datetime
df["order_date"] = pd.to_datetime(df["order_date"])

print(f"Total rows loaded : {len(df):,}")
print(f"Date range        : {df['order_date'].min().date()}  →  {df['order_date'].max().date()}")
df.head()



order_counts = df.groupby("customer_id")["order_date"].count()

# Keep only customers with >= 2 orders
repeat_customers = order_counts[order_counts >= 2].index
df_repeat = df[df["customer_id"].isin(repeat_customers)].copy()

print(f"Total unique customers    : {order_counts.shape[0]:,}")
print(f"Repeat customers (≥2 ord) : {len(repeat_customers):,}")
print(f"Rows kept                 : {len(df_repeat):,}")




# Sort by customer then chronologically
df_repeat = df_repeat.sort_values(["customer_id", "order_date"]).reset_index(drop=True)

# Calculate the gap between consecutive orders for the SAME customer
# .diff() on a datetime column returns a Timedelta; we extract the number of days
df_repeat["gap_days"] = (
    df_repeat.groupby("customer_id")["order_date"]
    .diff()                    # NaT for each customer's first order
    .dt.days                   # convert Timedelta → integer days
)

# Drop NaN rows (first order of every customer has no previous order to diff against)
gaps = df_repeat["gap_days"].dropna()

print(f"Total gap values (inter-order intervals): {len(gaps):,}")
print(f"Min gap : {gaps.min():.0f} days")
print(f"Max gap : {gaps.max():.0f} days")
df_repeat.head(10)




median_gap = gaps.median()

print("=" * 45)
print(f"  Median inter-order gap : {median_gap:.1f} days")
print("=" * 45)

Total rows loaded : 646,945
Date range        : 2012-07-04  →  2022-12-31
Total unique customers    : 90,246
Repeat customers (≥2 ord) : 67,888
Rows kept                 : 624,587
Total gap values (inter-order intervals): 556,699
Min gap : 0 days
Max gap : 3785 days
  Median inter-order gap : 144.0 days


In [2]:
import pandas as pd

df_products = pd.read_csv('../data/raw/products.csv', usecols=['segment', 'price', 'cogs'])

df_products['gross_margin'] = (df_products['price'] - df_products['cogs']) / df_products['price']

avg_margin_by_segment = df_products.groupby('segment')['gross_margin'].mean()

sorted_segments = avg_margin_by_segment.sort_values(ascending=False)
top_segment = sorted_segments.index[0]

print("Average Gross Margin by segment:")
print(sorted_segments.to_string())
print("\n" + "="*50)
print(f"Segment with the highest gross margin ratio: {top_segment}")
print("="*50)


Average Gross Margin by segment:
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Balanced       0.258038
Trendy         0.240758
Everyday       0.236343

Segment with the highest gross margin ratio: Standard


---
# Question 3: Most common return reason for the Streetwear category

In [3]:
import pandas as pd

df_returns = pd.read_csv('../data/raw/returns.csv', usecols=['product_id', 'return_reason'])
df_products = pd.read_csv('../data/raw/products.csv', usecols=['product_id', 'category'])

df_merged = pd.merge(df_returns, df_products, on='product_id', how='inner')
df_streetwear = df_merged[df_merged['category'] == 'Streetwear']

reason_counts = df_streetwear.groupby('return_reason')['product_id'].count()
top_reason = reason_counts.sort_values(ascending=False).index[0]

print("Count by reason:")
print(reason_counts.to_string())
print("\n" + "="*50)
print(f"Most common return reason: {top_reason}")
print("="*50)

Count by reason:
return_reason
changed_mind        3830
defective           4330
late_delivery       2159
not_as_described    3854
wrong_size          7626

Most common return reason: wrong_size


---
# Question 4: Traffic source with the lowest bounce rate

In [4]:
import pandas as pd

df_traffic = pd.read_csv('../data/raw/web_traffic.csv', usecols=['traffic_source', 'bounce_rate'])

bounce_by_source = df_traffic.groupby('traffic_source')['bounce_rate'].mean()
lowest_source = bounce_by_source.sort_values(ascending=True).index[0]

print("Average Bounce Rate:")
print(bounce_by_source.to_string())
print("\n" + "="*50)
print(f"Source with the lowest bounce rate: {lowest_source}")
print("="*50)

Average Bounce Rate:
traffic_source
direct            0.004511
email_campaign    0.004458
organic_search    0.004504
paid_search       0.004478
referral          0.004499
social_media      0.004476

Source with the lowest bounce rate: email_campaign


---
# Question 5: Percentage of lines with applied promotions

In [5]:
import pandas as pd

# Step 1: Load order_items table and count total rows
df_order_items = pd.read_csv('../data/raw/order_items.csv', usecols=['promo_id'])
total_rows = len(df_order_items)

# Step 2: Count lines with valid promotion codes (starting with 'PROMO-')
# Note: Need to handle NaN values before calling string methods (.str)
promo_rows = df_order_items['promo_id'].fillna('').str.startswith('PROMO-').sum()

# Step 3: Calculate percentage
promo_ratio = (promo_rows / total_rows) * 100

# Step 4: Print results for comparison
print(f"Total rows: {total_rows:,}")
print(f"Lines with valid promo codes: {promo_rows:,}")
print("="*50)
print(f"Promotion application rate: {promo_ratio:.2f}%")
print("="*50)

Total rows: 714,669
Lines with valid promo codes: 714,669
Promotion application rate: 100.00%


---
# Question 6: Age group with the highest average number of orders

In [6]:
import pandas as pd

# Step 1 & 2: Load customers table and drop empty age_group
df_customers = pd.read_csv('../data/raw/customers.csv', usecols=['customer_id', 'age_group'])
df_customers = df_customers.dropna(subset=['age_group'])

# Load orders table and count orders per customer
df_orders = pd.read_csv('../data/raw/orders.csv', usecols=['customer_id', 'order_id'])
order_counts = df_orders.groupby('customer_id')['order_id'].count().reset_index(name='order_count')

# Step 3: LEFT JOIN to ensure keeping customers who never purchased
df_merged = pd.merge(df_customers, order_counts, on='customer_id', how='left')

# Customers who never purchased will have Null order_count -> Fill with 0
df_merged['order_count'] = df_merged['order_count'].fillna(0)

# Step 4: Group by age_group and calculate average orders (total orders / total customers)
agg_df = df_merged.groupby('age_group').agg(
    total_orders=('order_count', 'sum'),
    total_customers=('customer_id', 'count')
)

agg_df['avg_orders'] = agg_df['total_orders'] / agg_df['total_customers']

# Step 5: Find age group with the highest average
top_age_group = agg_df['avg_orders'].sort_values(ascending=False).index[0]

print(agg_df)
print("\n" + "="*50)
print(f"Age group with the highest average number of orders: {top_age_group}")
print("="*50)

           total_orders  total_customers  avg_orders
age_group                                           
18-24           89057.0            17039    5.226656
25-34          190622.0            36342    5.245226
35-44          170368.0            31920    5.337343
45-54          124138.0            23172    5.357241
55+             72760.0            13457    5.406851

Age group with the highest average number of orders: 55+


---
# Question 7: Geographical region generating the highest total revenue

In [7]:
import pandas as pd

# Step 1: Load 3 tables, including order_status
df_orders = pd.read_csv('../data/raw/orders.csv', usecols=['order_id', 'zip', 'order_status'])
df_order_items = pd.read_csv('../data/raw/order_items.csv', usecols=['order_id', 'quantity', 'unit_price', 'discount_amount'])
df_geography = pd.read_csv('../data/raw/geography.csv', usecols=['zip', 'region'])

# Step 1.5: Remove cancelled orders
df_orders = df_orders[df_orders['order_status'] != 'cancelled']

# Step 2: Combine tables
df_merged = pd.merge(df_order_items, df_orders, on='order_id', how='inner')
df_merged = pd.merge(df_merged, df_geography, on='zip', how='inner')

# Step 3: Calculate net revenue for each line item
df_merged['discount_amount'] = df_merged['discount_amount'].fillna(0)
df_merged['revenue'] = (df_merged['quantity'] * df_merged['unit_price']) - df_merged['discount_amount']

# Step 4: Group by region and calculate total revenue
revenue_by_region = df_merged.groupby('region')['revenue'].sum()

# Step 5: Find the highest region
revenue_by_region = revenue_by_region.sort_values(ascending=False)
top_region = revenue_by_region.index[0]

print("Total revenue by region (excluding cancelled orders):")
print(revenue_by_region.apply(lambda x: f"${x:,.2f}").to_string())
print("\n" + "="*50)
print(f"Region with the highest total revenue: {top_region}")
print("="*50)

Total revenue by region (excluding cancelled orders):
region
East       $6,617,595,804.01
Central    $4,278,238,866.71
West       $3,338,020,957.28

Region with the highest total revenue: East


---
# Question 8: Payment methods for cancelled orders

In [8]:
import pandas as pd

df_orders = pd.read_csv('../data/raw/orders.csv', usecols=['order_status', 'payment_method'])
df_cancelled = df_orders[df_orders['order_status'] == 'cancelled']

payment_counts = df_cancelled.groupby('payment_method').size()
top_payment = payment_counts.sort_values(ascending=False).index[0]

print("Number of cancellations by payment method:")
print(payment_counts.to_string())
print("\n" + "="*50)
print(f"Most cancelled payment method: {top_payment}")
print("="*50)

Number of cancellations by payment method:
payment_method
apple_pay         5190
bank_transfer     2535
cod              15468
credit_card      28452
paypal            7817

Most cancelled payment method: credit_card


---
# Question 9: Size with the highest return rate

In [9]:
import pandas as pd

# Step 1: Load 3 tables
df_products = pd.read_csv('../data/raw/products.csv', usecols=['product_id', 'size'])
df_order_items = pd.read_csv('../data/raw/order_items.csv', usecols=['product_id'])
df_returns = pd.read_csv('../data/raw/returns.csv', usecols=['product_id'])

target_sizes = ['S', 'M', 'L', 'XL']

# Step 2: Calculate sold lines by size
sold_merged = pd.merge(df_order_items, df_products, on='product_id', how='inner')
sold_sizes = sold_merged[sold_merged['size'].isin(target_sizes)]
sold_counts = sold_sizes.groupby('size').size()

# Step 3: Calculate returned lines by size
returns_merged = pd.merge(df_returns, df_products, on='product_id', how='inner')
returns_sizes = returns_merged[returns_merged['size'].isin(target_sizes)]
returns_counts = returns_sizes.groupby('size').size()

# Step 4: Calculate Return Rate
return_rates = (returns_counts / sold_counts) * 100

# Step 5: Compare and find the highest result
return_rates = return_rates.sort_values(ascending=False)
top_size = return_rates.index[0]

print("Return Rate (based on lines) by size:")
print(return_rates.apply(lambda x: f"{x:.2f}%").to_string())
print("\n" + "="*50)
print(f"Size with the highest return rate: {top_size}")
print("="*50)

Return Rate (based on lines) by size:
size
S     5.65%
L     5.62%
M     5.57%
XL    5.52%

Size with the highest return rate: S


---
# Question 10: Installment plan with the highest average payment value

In [10]:
import pandas as pd

df_payments = pd.read_csv('../data/raw/payments.csv', usecols=['payment_value', 'installments'])

avg_payment = df_payments.groupby('installments')['payment_value'].mean()
top_installment = avg_payment.sort_values(ascending=False).index[0]

print("Average payment value by number of installments:")
print(avg_payment.apply(lambda x: f"${x:,.2f}").to_string())
print("\n" + "="*50)
print(f"Installment period with the highest average value: {top_installment} installments")
print("="*50)

Average payment value by number of installments:
installments
1     $24,113.27
2        $708.47
3     $24,399.64
6     $24,446.65
12    $24,245.77

Installment period with the highest average value: 6 installments
